### This notebook is used to test out the code before adding it to the partOne.py file

#### below is the code for read_novels function

In [ ]:
# %pip install pandas matplotlib nltk spacy

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import re   
import string ## to do fancy things with punctuation
import nltk
# nltk.download('punkt_tab')
# nltk.download("cmudict")

In [ ]:
file_path = Path(
    "/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw-pack-2026/texts/novels"
)
for f in file_path.glob("*.txt"):
    print(f)
    print(f.stem)
    title, author, year = f.stem.split("-")
    print(f"Title: {title}, Author: {author}, Year: {year}")

In [ ]:
# create a dataframe to store the metadata
rows = []
for f in file_path.glob("*.txt"):
    title, author, year = f.stem.split("-")
    text = f.read_text()
    rows.append({ "text": text, "title": title, "author": author, "year": year})

novels_df = pd.DataFrame(rows)
novels_df = novels_df.sort_values("year")
novels_df = novels_df.reset_index(drop=True)
novels_df.head(5)

In [ ]:
def read_novels(path=Path.cwd()/ "cw-pack-2026" / "texts" / "novels"):
    """Reads texts from a directory of .txt files and returns a DataFrame with the text, title,
    author, and year"""
    rows = []
    for f in path.glob("*.txt"):
        title, author, year = f.stem.split("-")
        text = f.read_text()
        rows.append({ "text": text, "title": title, "author": author, "year": year})
    novels_df = pd.DataFrame(rows)
    novels_df = novels_df.sort_values("year")
    novels_df = novels_df.reset_index(drop=True)
    return novels_df

In [ ]:
# check the function works
read_novels()

#### The code below develops the nltk_ttr function

In [ ]:
# first clean the text. No punctuations, all lowercase.
#  use one file as an example
text = novels_df.loc[0, "text"]
tokens = nltk.word_tokenize(text)
print(tokens[:20])
tokens = [t.lower() for t in tokens]
print(tokens[:20])
print(len(tokens))

# remove punctuation using regular expressions
re_punc = re.compile("[%s]" % re.escape(string.punctuation))
# substitute punctuation with empty string
tokens = [re_punc.sub("", t) for t in tokens]
print(tokens[:20])
print(len(tokens))

# only keep tokens that are not empty
tokens = [t for t in tokens if t.strip() != ""]
print(tokens[:20])
print(len(tokens))

# calculate type token ratio
ttr = len(set(tokens)) / len(tokens)
print(ttr)

In [ ]:
# define a clean text function
def clean_text(text):
    """Cleans a text by removing punctuation and converting to lowercase."""
    tokens = nltk.word_tokenize(text) # use nltk to tokenize the text into words
    tokens = [t.lower() for t in tokens]
    re_punc = re.compile("[%s]" % re.escape(string.punctuation))
    tokens = [re_punc.sub("", t) for t in tokens]
    tokens = [t for t in tokens if t.strip() != ""]
    return tokens

In [ ]:
# unit test the clean_text function
clean_text(text[0:50])

In [ ]:
# not in use for now. Email Paul to see if the text argument is necessary.
def nltk_ttr(text):
    """Calculates the type-token ratio of a text. Text is tokenized using nltk.word_tokenize."""
    # df = read_novels()
    path = Path.cwd()/ "cw-pack-2026" / "texts" / "novels"
    d_title2text = dict()
    d_text2ttr = dict()
    d_text2title = dict()
    d_text2ttr = dict()
    for f in path.glob("*.txt"):
        title, author, year = f.stem.split("-")
        title_text = f.read_text()
        d_title2text[title] = title_text
    
    tokens = clean_text(text)
    ttr = len(set(tokens)) / len(tokens)
    d_text2ttr[text] = ttr

    # convert d_title2text to d_text2title
    for k, v in d_title2text.items():
        d_text2title[v] = k
    
    d_text2ttr[d_text2title[text]] = d_text2ttr[text]
    return d_text2ttr

In [ ]:
# write a dictionary to map title to ttr
def nltk_ttr():
    """Calculates the type-token ratio of a text. Text is tokenized using nltk.word_tokenize."""
    # df = read_novels()
    path = Path.cwd()/ "cw-pack-2026" / "texts" / "novels"
    d_title2ttr = dict()
    for f in path.glob("*.txt"):
        title, author, year = f.stem.split("-")
        text = f.read_text()
        tokens = clean_text(text)
        ttr = len(set(tokens)) / len(tokens)
        d_title2ttr[title] = ttr

    return d_title2ttr  

In [ ]:
# unit test for nltk_ttr function
nltk_ttr()

### the following code is for flesch_kincaid function

In [ ]:
from nltk.corpus import cmudict
cmu = cmudict.dict()
def count_syllables(word):
    """Counts the number of syllables in a word using the cmudict."""
    phones = cmu.get(word.lower())
    if phones:
        return sum(1 for ph in phones[0] if ph[-1].isdigit())
    return None

In [ ]:
# unit test for nltk_ttr function
count_syllables("text")

In [ ]:
# count all syllables in the text
tokens = clean_text(text)
sum_syllables = 0
for t in tokens:
    syllables = count_syllables(t)
    if syllables is not None:
        sum_syllables += syllables
print(sum_syllables)

# define a function to count syllables in the text
def count_syllables_in_text(text):
    """Counts the total number of syllables in a text."""
    tokens = clean_text(text)
    sum_syllables = 0
    for t in tokens:
        syllables = count_syllables(t)
        if syllables is not None:
            sum_syllables += syllables
    return sum_syllables

In [ ]:
# unit test for count_syllables_in_text function
count_syllables_in_text(text)

In [ ]:
# define a function to count sentences in a text using nltk.sent_tokenize

def count_sentences(text):
    """Counts the number of sentences in a text using nltk.sent_tokenize."""
    sentences = nltk.sent_tokenize(text)
    return len(sentences)

In [ ]:
# unit test the count_sentences function
print(count_sentences(text))
sentences = "dfdfdf. sedretrefc. wegdsghg!"
print(count_sentences(sentences))

In [ ]:
# Flesch–Kincaid Grade Level = 0.39 * (total words / total sentences) 
# + 11.8 * (total syllables / total words) - 15.59
fk_grade = 0.39 * (len(tokens) / count_sentences(text)) + 11.8 * (sum_syllables / len(tokens)) - 15.59
print(fk_grade)

In [ ]:
def fk_level():
    """returns a diction mapping title to Flesch–Kincaid Grade Level."""
    path = Path.cwd()/ "cw-pack-2026" / "texts" / "novels"
    d_title2fk = dict()
    for f in path.glob("*.txt"):
        title, author, year = f.stem.split("-")
        text = f.read_text()
        tokens = clean_text(text)
        n_sentences = count_sentences(text)
        n_syllables = count_syllables_in_text(text)
        fk_grade = 0.39 * (len(tokens) / n_sentences) + 11.8 * (n_syllables / len(tokens)) - 15.59  
        d_title2fk[title] = fk_grade
    return d_title2fk

In [ ]:
# unit test for fk_level function
fk_level()

### The following code is to test functions regarding parse

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

novels_df.head(5)

In [ ]:
# apply nlp to each row in the dataframe and store the result in a new column called "Doc"
novels_df["Doc"] = novels_df["text"].apply(nlp)

display(novels_df.head(5))

In [ ]:
# serialise the df to disk using pickle
import pickle   
with open ("novels_df.pkl", "wb") as f:
    pickle.dump(novels_df, f)


In [ ]:
with open ("novels_df.pkl", "rb") as f:
    novels_df = pickle.load(f)  
display(novels_df.head(5))

In [ ]:
def parse(df, store_path=Path.cwd() / "pickles", out_name="parsed.pickle"):
    """Parses the text of a DataFrame using spaCy, stores the parsed docs as a column and writes 
    the resulting  DataFrame to a pickle file"""
    store_path.mkdir(parents=True, exist_ok=True)  # creates the folder if it doesn't exist
    df["Doc"] = df["text"].apply(nlp)
    with open (store_path / out_name, "wb") as f:
        pickle.dump(df, f)
    return df

In [ ]:
# unit test for parse function
parse(novels_df)

In [ ]:
with open ("novels_df.pkl", "rb") as f:
    novels_df = pickle.load(f)
display(novels_df.head(5))

### The code below test the code to answer question e (i)

In [ ]:
from collections import Counter

# syntaxtic subjects are nsubj (nominal subject) or nsubjpass (passive nominal subject) in spaCy's 
# _dep attribute. 

def get_subjects(doc):
    """Returns a list of the 10 most common syntactic subjects in a spaCy Doc."""
    subjects = []
    for token in doc:
        if token.dep_ in ("nsubj", "nsubjpass"):
            subjects.append(token.text.lower())
    
    counter = Counter(subjects)
    return counter.most_common(10)

In [ ]:
# loop through the df and list the titles and 10 most common syntactic subjects
for idx, row in novels_df.iterrows():
    title = row["title"]
    doc = row["Doc"]
    subjects = get_subjects(doc)
    print(f"Title: {title}, 10 most common syntactic subjects: {subjects}")

### The code below test the code to answer question e (ii)

In [ ]:
def get_pmi_verbs(doc, subject="he"):
    """return verbs most associated with the given subject, ordered by PMI"""
    he_verb_counts = Counter()
    all_subject_verb_counts = Counter()
    he_as_subject = 0 
    total_subject_tokens = 0

    for token in doc:
        if token.dep_ == "nsubj":
            verb = token.head.lemma_.lower()
            subj = token.text.lower()
            
            total_subject_tokens += 1
            all_subject_verb_counts[verb] += 1  # count for ALL subjects
            
            if subj == subject:
                he_as_subject += 1
                he_verb_counts[verb] += 1  # count only for "he"

    # calculate PMI for each verb that occurs with "he"
    pmi_scores = {}
    for verb, count in he_verb_counts.items():
        p_he_verb = count / total_subject_tokens
        p_he = he_as_subject / total_subject_tokens
        p_verb = all_subject_verb_counts[verb] / total_subject_tokens
        
        pmi = math.log(p_he_verb / (p_he * p_verb))
        pmi_scores[verb] = round(pmi, 4)

    sorted_pmi = sorted(pmi_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_pmi

In [ ]:
# write the loop
for idx, row in novels_df.iterrows():
    title = row["title"]
    doc = row["Doc"]
    pmi_verbs = get_pmi_verbs(doc, subject="he")
    print(f"Title: {title}, Verbs most associated with 'he' by PMI: {pmi_verbs[:10]}")

### The code below test the code to answer question e (iii)

In [ ]:
# write the loop
for idx, row in novels_df.iterrows():
    title = row["title"]
    doc = row["Doc"]
    pmi_verbs = get_pmi_verbs(doc, subject="she")
    print(f"Title: {title}, Verbs most associated with 'she' by PMI: {pmi_verbs[:10]}")